# Hindustani Raga Classifier — per-raga HMMs (Tansen-style)

**Why this exists**: real academic precedent for exactly this — Pandey, Mishra & Ipe (2003), *"Tansen: A System for Automatic Raga Identification"* (cited in the TRF paper's own literature review), models raga identification as one HMM per raga, trained generatively on that raga's note sequences, classified by Bayes' rule (which raga's HMM assigns highest posterior to this sequence). A different modeling paradigm from `raga_classifier_pitch_contour.ipynb`'s single discriminative GRU — worth comparing empirically, not assuming either is better.

**Why it might specifically suit our situation**: a set of permitted melodic transitions and characteristic phrases (arohana/avarohana, pakad) is close to literally what an HMM's states and transition matrix model. It also has far fewer parameters than even the shrunk GRU (61 independent small models vs. one 364K-param network) — continuing this project's running theme that data scarcity, not model capacity, has been the real constraint.

**Real caveat, not just an upside**: 61 *independent* per-class HMMs share zero parameters across classes, unlike the GRU's shared embedding/RNN — so while each HMM is individually tiny, the system as a whole isn't strictly more sample-efficient in aggregate. This is exactly the kind of thing to let the held-out numbers decide, not assume.

**No GPU needed** — `hmmlearn`'s Baum-Welch (EM) training is CPU-only and fast for this scale. Notebook Settings can skip the accelerator entirely, saving GPU quota.

**Reuses the exact same cached token sequences** as the pitch-contour GRU notebook — same quantization scheme, same config keys in the cache hash. If that notebook's run has completed, point `EXISTING_CACHE_INPUT` below at its Save Version output (Add Data → Notebook Output Files) to skip precompute entirely, same mechanism as before. If not, this notebook computes its own from scratch (self-sufficient either way).

**Saves per-segment log-likelihoods for all 61 classes on the test set** (not just the final accuracy) — needed later for the hybrid/ensemble idea (combine this model's and the GRU's predictions), once both have real standalone results to combine.

In [ ]:
# --- Config — cache-relevant values MUST match raga_classifier_pitch_contour.ipynb
# exactly for EXISTING_CACHE_INPUT reuse to work (same hash key formula). ---
AUDIO_SAMPLE_RATE = 44100
MELODY_HOP_SIZE = 128
QUANT_K = 5
VOCAB_SIZE = 209
INPUT_LENGTH = 5000
MIN_VOICED_FRACTION = 0.5
MAX_SEGMENTS_PER_FILE = 20
MAX_ANALYZE_SECONDS = 480
INTRO_SKIP_SECONDS = 20
OUTRO_SKIP_SECONDS = 15
PRECOMPUTE_WORKERS = 4

# HMM-specific config
N_HMM_STATES = 12    # rough analogue of 12 chromatic semitone positions per
                      # octave — a musically-motivated starting point, not
                      # tuned; states represent latent "functional" pitch
                      # categories, not the 209 fine-grained cents tokens directly
HMM_ITER = 100        # Baum-Welch (EM) iterations

SEED = 42             # SAME seed as both other notebooks — keeps the file-level
                      # split identical across all three for a fair comparison

SEGMENTS_CACHE_DIR = "/kaggle/working/pitch_cache"
MODELS_PATH = "/kaggle/working/hmm_models.pkl"

# Point at the pitch-contour GRU notebook's Save Version output (Add Data >
# Notebook Output Files) once that run has completed, to skip precompute
# entirely — same cache, same config, directly reusable. Leave None to
# compute fresh (self-sufficient, just slower).
EXISTING_CACHE_INPUT = None

In [ ]:
import subprocess, time, os

def run(cmd, timeout, label=None):
    label = label or cmd
    print(f"--- RUNNING ({timeout}s timeout): {label}")
    t0 = time.time()
    try:
        result = subprocess.run(cmd, shell=True, timeout=timeout,
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    except subprocess.TimeoutExpired as e:
        print(e.stdout or "")
        raise RuntimeError(f"TIMED OUT after {time.time()-t0:.0f}s: {label}")
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"FAILED (exit {result.returncode}): {label}")
    print(f"--- OK ({time.time()-t0:.0f}s): {label}")
    return result

run("apt-get -qq install -y ffmpeg", timeout=120, label="apt-get ffmpeg")
# essentia only actually needed if precompute runs from scratch (no cache
# attached) — installed unconditionally anyway since it's cheap and this
# notebook should work standalone. Same numpy-ABI defense applied proactively
# this time (learned the hard way in the GRU notebook, not repeating that mistake).
run("pip install -q essentia hmmlearn", timeout=300, label="pip install essentia + hmmlearn")
_pip_show = run("pip show numpy", timeout=30, label="read resolved numpy version")
_numpy_version = next(
    line.split(":", 1)[1].strip()
    for line in _pip_show.stdout.splitlines() if line.startswith("Version:")
)
print(f"Resolved numpy version: {_numpy_version}")
run(f"pip install -q --force-reinstall --no-deps numpy=={_numpy_version}", timeout=120,
    label="force-reinstall numpy binaries (pinned)")
run(
    'python -c "import numpy, sklearn, hmmlearn, essentia.standard as es; '
    'from hmmlearn import hmm; hmm.CategoricalHMM(n_components=4); '
    'es.TonicIndianArtMusic(); es.PredominantPitchMelodia(); '
    'print(numpy.__version__, hmmlearn.__version__, \'OK\')"',
    timeout=60,
    label="verify numpy/sklearn/hmmlearn/essentia all import and load cleanly",
)
os.makedirs(SEGMENTS_CACHE_DIR, exist_ok=True)

In [ ]:
import shutil

if EXISTING_CACHE_INPUT:
    if not os.path.isdir(EXISTING_CACHE_INPUT):
        raise RuntimeError(f"EXISTING_CACHE_INPUT={EXISTING_CACHE_INPUT!r} not found.")
    shutil.copytree(EXISTING_CACHE_INPUT, SEGMENTS_CACHE_DIR, dirs_exist_ok=True)
    print(f"Copied in {len(os.listdir(SEGMENTS_CACHE_DIR))} cached files from a prior session.")
else:
    print("No existing cache attached — starting precompute from scratch.")

In [ ]:
# --- Locate the dataset and enumerate every recording — identical to the
# other two notebooks, so all three are comparable. ---
import glob
from collections import defaultdict

candidates = [c for c in glob.glob("/kaggle/input/**/Thaat and Raga Forest*", recursive=True) if os.path.isdir(c)]
if not candidates:
    raise RuntimeError("TRF dataset not found under /kaggle/input. Add Data > 'suryamajumder/thaat-and-raga-forest-trf-dataset'.")
DATASET_ROOT = candidates[0]
print("Dataset root:", DATASET_ROOT)

files_by_raga = defaultdict(list)
for path in glob.glob(os.path.join(DATASET_ROOT, "*", "*", "*.mp3")):
    parts = path.split(os.sep)
    files_by_raga[(parts[-3], parts[-2])].append(path)

ragas = sorted(files_by_raga.keys(), key=lambda k: k[1])
raga2idx = {raga_key: i for i, raga_key in enumerate(ragas)}
idx2raga = {i: {"thaat": t, "raga": r} for (t, r), i in raga2idx.items()}
print(f"{len(ragas)} ragas, {sum(len(v) for v in files_by_raga.values())} recordings total")

In [ ]:
import random
random.seed(SEED)

train_files, val_files, test_files = [], [], []
for raga_key, fs in files_by_raga.items():
    fs = sorted(fs)
    random.shuffle(fs)
    label = raga2idx[raga_key]
    if len(fs) >= 3:
        test_files.append((fs[0], label))
        val_files.append((fs[1], label))
        train_files.extend((f, label) for f in fs[2:])
    else:
        train_files.extend((f, label) for f in fs)

all_files = train_files + val_files + test_files
print(f"train={len(train_files)}  val={len(val_files)}  test={len(test_files)}")

In [ ]:
# --- Precompute (only runs for files not already in the attached cache) —
# identical logic to raga_classifier_pitch_contour.ipynb, kept in sync so
# this notebook is self-sufficient even with no cache attached. ---
import numpy as np
import librosa
import hashlib
from concurrent.futures import ProcessPoolExecutor, as_completed

def _cache_path(path):
    config_key = (AUDIO_SAMPLE_RATE, MELODY_HOP_SIZE, QUANT_K, VOCAB_SIZE, INPUT_LENGTH,
                  MIN_VOICED_FRACTION, MAX_SEGMENTS_PER_FILE, MAX_ANALYZE_SECONDS,
                  INTRO_SKIP_SECONDS, OUTRO_SKIP_SECONDS)
    h = hashlib.md5(f"{path}|{config_key}".encode()).hexdigest()
    return os.path.join(SEGMENTS_CACHE_DIR, f"{h}.npy")

def _precompute_one(args):
    (path, sr, hop_size, quant_k, vocab_size, input_length, min_voiced_frac,
     max_segs, max_analyze_secs, intro_skip_secs, outro_skip_secs, out_path) = args
    if os.path.exists(out_path):
        return path, "cached", None
    try:
        import essentia.standard as es

        y, _ = librosa.load(path, sr=sr, mono=True)
        total_len = len(y)
        intro_skip = int(intro_skip_secs * sr)
        outro_skip = int(outro_skip_secs * sr)
        usable_start, usable_end = intro_skip, total_len - outro_skip
        if usable_end - usable_start < sr * 5:
            usable_start, usable_end = 0, total_len

        max_len = int(max_analyze_secs * sr)
        if usable_end - usable_start > max_len:
            usable_end = usable_start + max_len

        y = y[usable_start:usable_end].astype(np.float32)

        tonic = es.TonicIndianArtMusic(sampleRate=sr)(y)
        pitch, _conf = es.PredominantPitchMelodia(sampleRate=sr, hopSize=hop_size)(y)

        voiced = pitch > 0
        tokens = np.zeros(len(pitch), dtype=np.int32)
        cents = 1200.0 * np.log2(pitch[voiced] / tonic)
        tokens[voiced] = np.clip(np.round(cents * (quant_k / 100.0)), 0, vocab_size - 1).astype(np.int32)

        segments = []
        for s in range(0, max(1, len(tokens) - input_length + 1), input_length):
            chunk = tokens[s:s + input_length]
            if len(chunk) < input_length // 2:
                continue
            if voiced[s:s + len(chunk)].mean() < min_voiced_frac:
                continue
            if len(chunk) < input_length:
                chunk = np.pad(chunk, (0, input_length - len(chunk)))
            segments.append(chunk.astype(np.int16))
            if len(segments) >= max_segs:
                break

        if not segments:
            best_start, best_frac = 0, -1.0
            for s in range(0, max(1, len(tokens) - input_length + 1), input_length):
                frac = voiced[s:s + input_length].mean()
                if frac > best_frac:
                    best_frac, best_start = frac, s
            chunk = tokens[best_start:best_start + input_length]
            if len(chunk) < input_length // 2:
                return path, "empty", None
            if len(chunk) < input_length:
                chunk = np.pad(chunk, (0, input_length - len(chunk)))
            segments.append(chunk.astype(np.int16))

        np.save(out_path, np.stack(segments))
        return path, "done", None
    except Exception as e:
        return path, "error", str(e)

tasks = [(p, AUDIO_SAMPLE_RATE, MELODY_HOP_SIZE, QUANT_K, VOCAB_SIZE, INPUT_LENGTH,
          MIN_VOICED_FRACTION, MAX_SEGMENTS_PER_FILE, MAX_ANALYZE_SECONDS,
          INTRO_SKIP_SECONDS, OUTRO_SKIP_SECONDS, _cache_path(p)) for p, _ in all_files]
already_cached = sum(1 for t in tasks if os.path.exists(t[-1]))
print(f"{already_cached}/{len(tasks)} already cached (resumed) — {len(tasks) - already_cached} left to process")

t0 = time.time()
done, errors = 0, []
with ProcessPoolExecutor(max_workers=PRECOMPUTE_WORKERS) as ex:
    futures = [ex.submit(_precompute_one, t) for t in tasks]
    for fut in as_completed(futures):
        path, status, err = fut.result()
        if status == "error":
            errors.append((path, err))
            print(f"  ERROR on {path}: {err}")
        done += 1
        if done % 20 == 0 or done == len(tasks):
            elapsed = time.time() - t0
            rate = elapsed / max(1, done - already_cached) if done > already_cached else None
            remaining = len(tasks) - done
            eta = f"{rate * remaining / 60:.1f}min" if rate else "n/a (still warming up)"
            print(f"  {done}/{len(tasks)} processed, {elapsed/60:.1f}min elapsed, ETA for rest: {eta}")

print(f"Precompute finished: {done} processed, {len(errors)} errors.")
if errors:
    print("Files with errors (will be skipped below):", [e[0] for e in errors])

In [ ]:
# --- Train one CategoricalHMM per raga on its own training segments
# (Tansen-style). hmmlearn wants a single concatenated observation array
# plus a `lengths` list marking where each individual sequence starts.
#
# Per-model progress + a per-class segment cap added 2026-09-17 after a live
# run went over 20 minutes with zero output: models train in raga-label
# order, and Baum-Welch cost scales with TOTAL observations (segments x
# INPUT_LENGTH) for that class alone — a raga with ~40 recordings x 20
# segments/file = 800 segments x 5000 tokens = 4M+ observations for ONE
# fit. Progress was only printed every 10 *completed* models, so one slow
# early class (in label order) could stay silent a long time with no way to
# tell it wasn't hung. Fixed two ways: print before/after EVERY model (not
# batched), and train smallest classes first (fast, frequent progress lines
# early) with a hard cap on segments/class so no single huge raga can
# dominate total runtime (subsampled reproducibly via SEED, not truncated
# from the start, so the cap doesn't bias toward one part of the training set). ---
from hmmlearn import hmm
from collections import defaultdict as _dd

MAX_SEGMENTS_PER_CLASS = 300  # bounds worst-case Baum-Welch cost per model

def _load_cached(path):
    cache_path = _cache_path(path)
    if not os.path.exists(cache_path):
        return None
    return np.load(cache_path)

train_segments_by_class = _dd(list)  # label -> list of 1D int arrays
skipped = 0
for path, label in train_files:
    segs = _load_cached(path)
    if segs is None:
        skipped += 1
        continue
    for seg in segs:
        train_segments_by_class[label].append(seg.astype(np.int64))
if skipped:
    print(f"(skipping {skipped} train files with no cache)")

total_train_segments = sum(len(v) for v in train_segments_by_class.values())
print(f"{total_train_segments} total training segments across {len(train_segments_by_class)} ragas")

# Class priors computed from TRUE segment counts (before any capping below),
# so the Bayes decision rule still reflects the dataset's real imbalance.
class_priors = {
    label: len(segs) / total_train_segments
    for label, segs in train_segments_by_class.items()
}

rng = random.Random(SEED)
capped = 0
for label, segs in train_segments_by_class.items():
    if len(segs) > MAX_SEGMENTS_PER_CLASS:
        train_segments_by_class[label] = rng.sample(segs, MAX_SEGMENTS_PER_CLASS)
        capped += 1
if capped:
    print(f"Capped {capped} ragas to {MAX_SEGMENTS_PER_CLASS} segments each (reproducibly subsampled) "
          f"to bound worst-case per-model training time.")

# Smallest classes first — fast, frequent progress lines right away rather
# than possibly sitting on the biggest, slowest class first with nothing to show.
class_order = sorted(train_segments_by_class.items(), key=lambda kv: len(kv[1]))

models = {}
t_total = time.time()
for i, (label, segs) in enumerate(class_order):
    raga_name = idx2raga[label]["raga"]
    print(f"[{i+1}/{len(class_order)}] training '{raga_name}' "
          f"({len(segs)} segments, {len(segs)*INPUT_LENGTH:,} observations)...", flush=True)
    t0 = time.time()
    X = np.concatenate(segs).reshape(-1, 1)
    lengths = [len(s) for s in segs]
    model = hmm.CategoricalHMM(n_components=N_HMM_STATES, n_iter=HMM_ITER,
                                random_state=SEED, n_features=VOCAB_SIZE)
    try:
        model.fit(X, lengths)
        models[label] = model
        print(f"  -> done in {time.time()-t0:.0f}s  (total elapsed: {(time.time()-t_total)/60:.1f}min)", flush=True)
    except Exception as e:
        print(f"  -> WARNING: failed to fit ({time.time()-t0:.0f}s): {e}", flush=True)

print(f"Done: {len(models)}/{len(ragas)} raga HMMs trained successfully in {(time.time()-t_total)/60:.1f}min")

import pickle
with open(MODELS_PATH, "wb") as f:
    pickle.dump({"models": models, "class_priors": class_priors, "idx2raga": idx2raga}, f)
print(f"Saved trained models -> {MODELS_PATH}")

In [ ]:
# --- Held-out TEST evaluation: per-segment AND per-file majority-vote
# accuracy, matching the other two notebooks' methodology exactly so all
# three are directly comparable. Classification = Bayes rule: argmax over
# (log-likelihood under model_k + log(prior_k)). Also saves the full
# per-segment log-likelihood matrix for later use in an ensemble with the
# GRU's predictions. ---
import math

log_priors = {label: math.log(p) for label, p in class_priors.items()}
fitted_labels = sorted(models.keys())

per_class_correct = defaultdict(int)
per_class_total = defaultdict(int)
seg_correct, seg_total = 0, 0
file_correct, file_total = 0, 0
all_test_loglik = []   # list of (file_path, true_label, {label: loglik per segment})

t0 = time.time()
for path, label in test_files:
    segs = _load_cached(path)
    if segs is None:
        continue
    seg_preds = []
    seg_logliks = []
    for seg in segs:
        x = seg.astype(np.int64).reshape(-1, 1)
        scores = {k: models[k].score(x) + log_priors[k] for k in fitted_labels}
        seg_logliks.append(scores)
        pred = max(scores, key=scores.get)
        seg_preds.append(pred)
        seg_total += 1
        if pred == label:
            seg_correct += 1
    all_test_loglik.append((path, label, seg_logliks))

    majority = max(set(seg_preds), key=seg_preds.count)
    file_total += 1
    per_class_total[label] += 1
    if majority == label:
        file_correct += 1
        per_class_correct[label] += 1

print(f"Scored {file_total} test files in {time.time()-t0:.0f}s")
print(f"Per-segment test accuracy: {seg_correct/max(1,seg_total):.3f}  ({seg_correct}/{seg_total})")
print(f"Per-file majority-vote test accuracy: {file_correct/max(1,file_total):.3f}  ({file_correct}/{file_total})")
print()
print("Per-class (majority-vote) breakdown:")
for label in sorted(per_class_total):
    print(f"  {idx2raga[label]['raga']:35s} {per_class_correct[label]}/{per_class_total[label]}")

In [ ]:
import json

with open("/kaggle/working/hmm_test_logliks.pkl", "wb") as f:
    pickle.dump(all_test_loglik, f)

with open("/kaggle/working/hmm_results.json", "w") as f:
    json.dump({
        "test_segment_acc": seg_correct / max(1, seg_total),
        "test_majority_vote_acc": file_correct / max(1, file_total),
        "num_ragas": len(ragas),
        "num_fitted_models": len(models),
        "n_hmm_states": N_HMM_STATES,
    }, f, indent=2)

print("Saved: hmm_models.pkl (61 trained HMMs + priors), hmm_test_logliks.pkl "
      "(per-segment log-likelihoods, for a later ensemble with the GRU), hmm_results.json")
print("Click 'Save Version' now to persist these for comparison / reuse.")